# 02｜Return、GAE 与 PPO 裁剪

对应[第 02 章](../course/02-mdp-reward-and-ppo.md)。这里逐项观察数值；之后仍需独立完成 [`02_advantage_exercise.py`](../labs/starter/02_advantage_exercise.py)。

## 学习目标

区分即时 reward、折扣 return、value 和 advantage；理解 done mask；从 log-prob 得到概率比；不用背表即可推导 PPO 正负优势的裁剪方向。

## 本节知识地图

本节只学 6 个知识点。它们是一条计算链，不是六个孤立定义。

| 知识点 | 一句话解释 | 在训练中对应什么 | 掌握检查 |
| --- | --- | --- | --- |
| reward $r_t$ | 当前一步环境给的学习信号 | Panda reward terms | 不把它叫成功率 |
| return $G_t$ | 从 t 开始的折扣累计 reward | critic 的学习目标 | 手算两步 return |
| value $V(s_t)$ | 状态未来 return 的估计 | value network | 说出它为何是 baseline |
| advantage $A_t$ | 动作结果相对 baseline 好多少 | policy update 权重 | 区分正负含义 |
| GAE | 从后向前混合多步 TD residual | advantage estimator | 解释 done mask |
| PPO clipping | 限制新旧策略概率比的有利过度更新 | clipped objective | 推导正负优势各一例 |

## 关键概念与符号

| 符号 | 含义 | 先记住什么 |
| --- | --- | --- |
| $\gamma$ | 未来 reward 的折扣 | 越大越重视远期 |
| $\lambda$ | GAE 多步混合强度 | 不是学习率 |
| $d_t$ | 该步是否终止 | 终止后不 bootstrap 下一回合 |
| $\delta_t$ | 一步 TD residual | reward + 下一 value − 当前 value |
| ratio | `exp(new_log_prob-old_log_prob)` | 是动作概率比，不是动作幅度比 |
| $\epsilon$ | PPO 裁剪宽度 | 不是探索概率 |

> **不要混淆：** reward ≠ return；value ≠ advantage；动作数值 ≠ 动作的策略概率；PPO clipping 约束更新幅度，但不能修复错误 reward。

## 先预测

给定 `reward=[0,1]`、`value=[0.4,0.6,0]`、第二步结束、$\gamma=0.9$、$\lambda=0.95$：

1. 两步 return 是多少？
2. 两个 TD residual 是多少？
3. `A=+1, ratio=1.5, epsilon=0.2` 的 PPO 样本目标是多少？
4. `A=-1, ratio=0.5` 为什么不能沿用正优势的直觉？

## 运行与观察

先加载教学函数并验证项目 kernel。函数是可读的数值推导；正式 PPO 更新仍由 Brax 实现。

In [ ]:
from pathlib import Path
import sys

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'pyproject.toml').is_file())
sys.path.insert(0, str(ROOT / 'docs' / 'notebooks'))

import matplotlib.pyplot as plt
import numpy as np
from course_utils import assert_course_kernel, clipped_objective, discounted_returns, generalized_advantage

assert_course_kernel(ROOT)
print('Kernel:', sys.executable)

### 1. Reward 不等于 return

return 从轨迹末端向前累计；done 后不得把下一回合奖励接进当前回合。下面打印每个时间步，不要只看最终数组。

In [ ]:
rewards = np.array([0.0, 1.0])
dones = np.array([0.0, 1.0])
gamma = 0.9
returns = discounted_returns(rewards, dones, gamma)
for t, (reward, done, ret) in enumerate(zip(rewards, dones, returns)):
    print(f't={t} reward={reward:.1f} done={done:.0f} return={ret:.3f}')
assert np.allclose(returns, [0.9, 1.0])

### 2. Value 是 baseline，GAE 是相对判断

TD residual 为 $\delta_t=r_t+\gamma(1-d_t)V_{t+1}-V_t$。GAE 再从后向前混合 residual。

In [ ]:
values = np.array([0.4, 0.6, 0.0])  # 比 reward 多一个 bootstrap value
gae_lambda = 0.95
active = 1.0 - dones
deltas = rewards + gamma * active * values[1:] - values[:-1]
advantages = generalized_advantage(rewards, values, dones, gamma, gae_lambda)
for t in range(len(rewards)):
    print(f't={t} delta={deltas[t]:.3f} GAE={advantages[t]:.3f}')
assert np.allclose(deltas, [0.14, 0.4])
assert np.allclose(advantages, [0.482, 0.4])

### 3. 从 log-prob 到 ratio

`ratio = exp(new_log_prob - old_log_prob)`。它描述同一个已采样动作的相对概率变化，不是动作幅度或 reward 的比值。

In [ ]:
old_log_prob = -2.0
new_log_prob = -1.8
ratio_from_logs = np.exp(new_log_prob - old_log_prob)
print(f'probability ratio = {ratio_from_logs:.4f} ({(ratio_from_logs - 1):.1%} relative change)')
assert np.isclose(ratio_from_logs, np.exp(0.2))

### 4. 可视化 PPO conservative minimum

实线是最终样本目标，灰色虚线是未裁剪目标。正优势限制右侧过度增加；负优势限制左侧过度减少。PPO 不会裁掉让策略变差的方向。

In [ ]:
ratios = np.linspace(0.4, 1.6, 241)
epsilon = 0.2
fig, axes = plt.subplots(1, 2, figsize=(11, 4), sharex=True)
for ax, advantage in zip(axes, (1.0, -1.0)):
    adv = np.full_like(ratios, advantage)
    objective = clipped_objective(ratios, adv, epsilon)
    ax.plot(ratios, ratios * advantage, '--', color='0.55', label='unclipped')
    ax.plot(ratios, objective, color='tab:blue', label='PPO objective')
    ax.axvspan(1-epsilon, 1+epsilon, alpha=0.12, color='tab:green', label='clip interval')
    ax.axvline(1, color='0.25', linewidth=1)
    ax.set(title=f'advantage = {advantage:+.0f}', xlabel='new / old probability ratio', ylabel='sample objective')
    ax.legend()
plt.tight_layout(); plt.show()

## 动手修改

把 `epsilon_project` 从 0.2 改为本项目视觉 PPO 使用的 0.3。先预测四格中哪些数值变化，再运行。最后恢复 0.3 供自测使用。

In [ ]:
epsilon_project = 0.3
four_ratios = np.array([1.5, 0.5, 0.5, 1.5])
four_advantages = np.array([1.0, 1.0, -1.0, -1.0])
four_objectives = clipped_objective(four_ratios, four_advantages, epsilon_project)
for ratio, advantage, objective in zip(four_ratios, four_advantages, four_objectives):
    print(f'ratio={ratio:.1f} advantage={advantage:+.0f} objective={objective:+.2f}')

## 自测

先解释为什么第二项和第四项保留未裁剪的较差值，再运行断言。

In [ ]:
assert np.allclose(four_objectives, [1.3, 0.5, -0.7, -1.5])
assert returns[0] != rewards[0]
assert advantages.shape == rewards.shape
assert dones[-1] == 1 and values[-1] == 0
print('PASS: return, done mask, GAE, log-prob ratio, and PPO clipping')

## 学完请记住

关闭本页后，你应能脱稿说出：

1. reward 是一步信号，return 是未来累计，value 是其估计；
2. advantage 为正表示该动作比 baseline 好，为负表示更差；
3. done mask 阻止下一回合 value 污染当前回合；
4. ratio 比较新旧策略对同一已采样动作的概率；
5. PPO 取 conservative minimum，因此正负优势的裁剪方向不同；
6. PPO 是 on-policy 强化学习，不是模仿学习。

若无法从 reward 推到 PPO 样本目标，按 1→5 的顺序定位断点，不要直接背公式。

## 反思与记录

在 `notes/02-notebook-reflection.md` 回答：

1. value network 与部署时 deterministic policy 分别做什么？
2. 为什么 done mask 缺失会把下一回合污染进当前回合？
3. PPO 裁剪为何不能修复错误 reward 或 guide-state 评估泄漏？
4. `num_updates_per_batch=8` 为什么不是产生 8 倍新环境数据？

然后独立实现 starter；不要从本 Notebook 复制函数。